In [1]:
import os
import sys

import torch
import numpy as np
import pandas as pd

import shared.utils as su

In [2]:
from transformers import Qwen2_5_VLModel, Qwen2_5_VLForConditionalGeneration
from models.qwen25vl import EncoderForQwen25VL

In [8]:
model_path = "/work/piyush/pretrained_checkpoints/ArrowRL-Qwen2.5-VL-7B"
model = EncoderForQwen25VL.from_pretrained(model_path, torch_dtype=torch.float16, attn_implementation='flash_attention_2', device_map='auto')
su.misc.num_params(model.model)

Loading EncoderForQwen25VL from /work/piyush/pretrained_checkpoints/ArrowRL-Qwen2.5-VL-7B


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

::: Number of total parameters in Qwen2_5_VLForConditionalGeneration: 8292.167M


In [4]:
def embed_video_(path):
    return model.encode_vision([path]).squeeze(0)


embed_video_("../assets/demo.mp4").shape

qwen-vl-utils using torchcodec to read video.


torch.Size([3584])

In [5]:
def embed_text_(text):
    return model.encode_text(text).squeeze(0)


embed_text_("Sample").shape

torch.Size([3584])

In [10]:
model.encode_vision_text("../assets/demo.mp4", "Sample edit text").shape

torch.Size([1, 3584])

### WebVid-CoVR

In [19]:
def load_data(dataset: str = 'covr') -> pd.DataFrame:
    """
    Load dataset configuration and CSV file for CoVR.
    
    Args:
        dataset: Name of dataset to load (covr)
        
    Returns:
        DataFrame with video paths and metadata
    """
    import json
    
    # Load dataset config from YAML
    cfg_path = "../datasets.yaml"
    assert os.path.exists(cfg_path), f"Dataset config file {cfg_path} does not exist"
    all_configs = su.io.load_yml(cfg_path)
    
    # Validate dataset
    assert dataset in all_configs, f"Dataset {dataset} not found in datasets.yaml"
    
    data_config = all_configs[dataset]
    
    # Load CSV
    csv_path = data_config['csv_path']
    video_dir = data_config['video_dir']
    
    assert os.path.exists(csv_path), f"CSV file {csv_path} does not exist"
    df = pd.read_csv(csv_path)
    
    print(f"Dataset: {dataset}")
    print(f"Number of rows in CoVR-test: {len(df)}")
    
    # Add video paths
    df['video1_path'] = df['video1'].apply(lambda x: f"{video_dir}/{x}")
    df['video2_path'] = df['video2'].apply(lambda x: f"{video_dir}/{x}")
    
    # Filter to only existing videos
    df = df[df.video1_path.apply(os.path.exists) & df.video2_path.apply(os.path.exists)]
    print(f"Number of rows with all videos available: {df.shape}")
    
    # Remove problematic videos (if any)
    problematic_videos = data_config.get('problematic_videos', [])
    for video_path in problematic_videos:
        df = df[df.video1_path != video_path]
    
    print(f"Sample row: ")
    print(json.dumps(df.iloc[0].to_dict(), indent=4))

    return df

In [22]:
df = load_data(dataset="covr")
# Load dataset config
cfg_path = "../datasets.yaml"
all_configs = su.io.load_yml(cfg_path)
data_config = all_configs["covr"]
video_dir = data_config['video_dir']
print('-' * 100)


Dataset: covr
Number of rows in CoVR-test: 2556
Number of rows with all videos available: (2556, 11)
Sample row: 
{
    "txt1": "Digital network, white lines and dots",
    "txt2": "Digital network, green lines and dots",
    "sim_txt": 0.8944025,
    "pth1": "112/1016223889",
    "pth2": "112/1016223877",
    "edit": "replace the white lines and dots with green",
    "scores": "[0.3524, 0.4087, 0.4136, 0.4111, 0.4137, 0.4084, 0.4062, 0.4054, 0.4025, 0.413, 0.4115, 0.4105, 0.4123, 0.4099, 0.4107]",
    "video1": "070651_070700/1016223889.mp4",
    "video2": "032701_032750/1016223877.mp4",
    "video1_path": "/datasets/WebVid/videos/070651_070700/1016223889.mp4",
    "video2_path": "/datasets/WebVid/videos/032701_032750/1016223877.mp4"
}
----------------------------------------------------------------------------------------------------


In [24]:
# Compute video embeddings for the candidate videos
videos = set(df.video2.tolist())
candidates = {}
for video in su.log.tqdm_iterator(videos, desc='Computing features for candidate videos'):
    video_path = f"{video_dir}/{video}"
    assert os.path.exists(video_path)
    with torch.no_grad():
        zv = model.encode_vision(video_path).squeeze(0)
        zv = torch.nn.functional.normalize(zv, dim=-1)
        zv = zv.cpu().float()
    candidates[video] = zv
print(f"Successfully computed {len(candidates)} candidate embeddings.")


Computing features for candidate videos:   0%|          | 0/2555 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
# Gather query embeddings
query_embeds = {}
for i in su.log.tqdm_iterator(range(len(query_embeds), len(df)), desc="Compute query embeddings"):
    row = df.iloc[i].to_dict()
    video_path = f"{video_dir}/{row['video1']}"
    edit_text = row['edit']
    with torch.no_grad():
        try:
            zv = model.encode_vision_text(video_path, edit_text).squeeze(0)
            zv = torch.nn.functional.normalize(zv, dim=-1)
            zv = zv.cpu().float()
            key = f"{edit_text}|{row['video1']}"
            query_embeds[key] = zv
        except:
            print(f"Skpping {i}")
            continue
print(f"Successfully computed {len(query_embeds)} query embeddings.")

Compute query embeddings:   0%|          | 0/2555 [00:00<?, ?it/s]

Skpping 3


### RTime

In [10]:
from evals_vlm2vec2.rtime import *

In [22]:
# Load RTime data
data_dir = "/scratch/shared/beegfs/piyush/datasets/ReversedInTime"
csv_path = f"{data_dir}/splits/all_meta.csv"
df = pd.read_csv(csv_path)
df = df[df.split == 'test']
df.video_id = df.video_id.astype(str)
data = load_json(f"{data_dir}/splits/test.json")
len(data)

1000

In [ ]:
# Compute embeddings
norm = lambda x: torch.nn.functional.normalize(x, dim=-1).cpu().float()
vid_fwd_emb, vid_rev_emb = {}, {}
cap_fwd_emb, cap_rev_emb = {}, {}

for i in tqdm_iterator(range(len(df)), desc='Computing RTime embeddings'):
    row = df.iloc[i].to_dict()
    video_id = str(row['video_id'])

    try:
        vid_fwd_path = f"{data_dir}/videos/{video_id}.mp4"
        vid_rev_path = f"{data_dir}/videos/{video_id}-reverse.mp4"
        assert os.path.exists(vid_fwd_path)
        assert os.path.exists(vid_rev_path)

        cap_fwd_text = data[video_id]['forward_captions'][0]
        cap_rev_text = data[video_id]['reverse_captions'][0]

        with torch.no_grad():
            vid_fwd = norm(embed_video_(vid_fwd_path))
            vid_rev = norm(embed_video_(vid_rev_path))
            cap_fwd = norm(embed_text_(cap_fwd_text))
            cap_rev = norm(embed_text_(cap_rev_text))

        vid_fwd_emb[video_id] = vid_fwd
        vid_rev_emb[video_id] = vid_rev
        cap_fwd_emb[video_id] = cap_fwd
        cap_rev_emb[video_id] = cap_rev
    except:
        print(f"Error computing embeddings for {video_id}")
        continue

df = df[df.video_id.isin(list(vid_fwd_emb.keys()))]
print(f"Number of rows with valid embeddings: {len(df)}")


Computing RTime embeddings:   0%|          | 0/1000 [00:00<?, ?it/s]

Error computing embeddings for 18599216


In [ ]:
# Compute metrics
metrics = {
    'binary': compute_rtime_binary_score(vid_fwd_emb, vid_rev_emb, cap_fwd_emb, cap_rev_emb),
    'origin': compute_rtime_origin_score(vid_fwd_emb, cap_fwd_emb),
    'hard': compute_rtime_hard_score(vid_fwd_emb, vid_rev_emb, cap_fwd_emb, cap_rev_emb),
}
print_metrics(metrics)

# Save metrics
result_dir = os.path.join(model_path, "metrics")
os.makedirs(result_dir, exist_ok=True)
suffix = "all"
model_name = "qwen25vl_arrowrl"
metrics_path = os.path.join(result_dir, f"metrics_rtime_{model_name}_{suffix}.json")
with open(metrics_path, "w") as f:
    json.dump(metrics, f, indent=4)
print(f"Saved metrics to {metrics_path}")